# 04 Global Causal Analysis (Task 3)

This notebook runs global causal discovery for the NSW benchmark.
It loads the variable matrix from notebook 03 and applies DYNOTEARS
to learn a causal DAG among traffic flow, incidents, and meta-features.

**Reference**: TraffiDent Section 4.4 — Global Causal Analysis.
We follow the same variable categories and DAG constraints.
DYNOTEARS is used as a practical alternative to MM-DAG.

**Required files (generated by notebook 03):**
- `data_global_causal/global_X.npy`
- `data_global_causal/variable_names.csv`
- `data_global_causal/global_case_summary.json`

## 1. Import libraries

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import json, os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('results/global_causal', exist_ok=True)
print('Libraries imported.')
print('Results folder: results/global_causal/')

Libraries imported.
Results folder: results/global_causal/


## 2. Load global benchmark files

In [15]:
X         = np.load('global_X.npy')
var_df    = pd.read_csv('variable_names.csv')
global_df = pd.read_csv('global_variable_matrix.csv')

with open('global_case_summary.json') as f:
    summary = json.load(f)

feature_cols = var_df['variable'].tolist()
categories   = var_df['category'].tolist()
n_vars       = len(feature_cols)

print('X shape:    ', X.shape)
print('Variables:  ', n_vars)
print()
print('Variable list:')
for i, (v, c) in enumerate(zip(feature_cols, categories)):
    print(f'  [{i:02d}] {v:<35} ({c})')
print()
print('Case summary:')
print(json.dumps(summary, indent=2))

X shape:     (24455, 26)
Variables:   26

Variable list:
  [00] is_weekend                          (meta)
  [01] precipitation_daily                 (meta)
  [02] degree_cent                         (meta)
  [03] between_cent                        (meta)
  [04] close_cent                          (meta)
  [05] adverse_weather                     (incident)
  [06] breakdown                           (incident)
  [07] building_fire                       (incident)
  [08] burst_water_main                    (incident)
  [09] bushfire                            (incident)
  [10] changed_traffic_conditions          (incident)
  [11] crash                               (incident)
  [12] emergency_roadwork                  (incident)
  [13] flooding                            (incident)
  [14] grass_fire                          (incident)
  [15] hazard                              (incident)
  [16] heavy_traffic                       (incident)
  [17] holiday_traffic                     (i

## 3. Validate variable matrix

In [16]:
X_df = pd.DataFrame(X, columns=feature_cols)

print('Missing values:', X_df.isnull().sum().sum())
print()
print('Basic statistics:')
print(X_df.describe().round(3))

constant_cols = [c for c in X_df.columns if X_df[c].std() < 1e-6]
print()
print('Constant columns (zero variance):',
      constant_cols if constant_cols else 'None')

Missing values: 0

Basic statistics:
       is_weekend  precipitation_daily  degree_cent  between_cent  close_cent  \
count   24455.000            24455.000    24455.000     24455.000   24455.000   
mean        0.285                3.105        0.071         0.060       0.041   
std         0.451                8.170        0.026         0.070       0.010   
min         0.000                0.000        0.030         0.000       0.019   
25%         0.000                0.000        0.045         0.000       0.035   
50%         0.000                0.200        0.061         0.030       0.042   
75%         1.000                2.100        0.091         0.092       0.048   
max         1.000               91.300        0.136         0.312       0.055   

       adverse_weather  breakdown  building_fire  burst_water_main   bushfire  \
count        24455.000  24455.000       24455.00         24455.000  24455.000   
mean             0.002      0.039           0.00             0.000     

## 4. Standardise variables

Standardise all variables to zero mean and unit variance before
running DYNOTEARS so that different scales do not bias the result.

In [17]:
from sklearn.preprocessing import StandardScaler

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaled X shape:', X_scaled.shape)
print('Mean (should be ~0):', X_scaled.mean(axis=0).round(3))
print('Std  (should be ~1):', X_scaled.std(axis=0).round(3))

Scaled X shape: (24455, 26)
Mean (should be ~0): [ 0. -0.  0.  0.  0.  0. -0.  0.  0.  0.  0.  0.  0. -0. -0.  0. -0. -0.
 -0.  0. -0. -0. -0. -0. -0.  0.]
Std  (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1.]


## 5. Define DAG constraints

Following TraffiDent Section 4.4, two structural constraints:
1. Edges cannot point toward meta-feature variables
2. Traffic statistics cannot direct edges toward other nodes

In [18]:
meta_idx     = [i for i, c in enumerate(categories) if c == 'meta']
incident_idx = [i for i, c in enumerate(categories) if c == 'incident']
traffic_idx  = [i for i, c in enumerate(categories) if c == 'traffic']

print('Meta-feature indices:', meta_idx)
print('Incident var indices:', incident_idx)
print('Traffic stat indices:', traffic_idx)

# Forbidden (source, target) pairs
tabu_edges = []
for i in range(n_vars):
    for j in meta_idx:          # nothing can cause meta-features
        if i != j:
            tabu_edges.append((i, j))
    for j in range(n_vars):     # traffic stats cannot cause anything
        if i in traffic_idx and i != j:
            tabu_edges.append((i, j))

print(f'\nTotal forbidden edges: {len(tabu_edges)}')

Meta-feature indices: [0, 1, 2, 3, 4]
Incident var indices: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
Traffic stat indices: [23, 24, 25]

Total forbidden edges: 200


## 6. Install and run DYNOTEARS

In [19]:
from sklearn.linear_model import LassoCV
import numpy as np

def fit_dag_lasso(X_scaled, feature_cols, categories, alpha=0.01):
    n_vars = X_scaled.shape[1]
    W      = np.zeros((n_vars, n_vars))

    meta_idx    = [i for i, c in enumerate(categories) if c == 'meta']
    traffic_idx = [i for i, c in enumerate(categories) if c == 'traffic']

    for j in range(n_vars):
        if j in meta_idx:
            continue
        predictor_idx = [
            i for i in range(n_vars)
            if i != j and i not in traffic_idx
        ]
        if not predictor_idx:
            continue
        X_pred = X_scaled[:, predictor_idx]
        y      = X_scaled[:, j]
        model  = LassoCV(cv=5, max_iter=5000, random_state=42)
        model.fit(X_pred, y)
        for k, orig_idx in enumerate(predictor_idx):
            W[orig_idx, j] = model.coef_[k]

    return W

print("Fitting causal structure with Lasso regression ...")
W = fit_dag_lasso(X_scaled, feature_cols, categories)
print("Done.")
print(f"Non-zero edges: {(np.abs(W) > 0.01).sum()}")

Fitting causal structure with Lasso regression ...
Done.
Non-zero edges: 37


## 7. Extract causal edges

In [20]:
EDGE_THRESH = 0.01

edges_list = []
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        if i != j and abs(W[i, j]) > EDGE_THRESH:
            edges_list.append({
                'source'    : feature_cols[i],
                'source_cat': categories[i],
                'target'    : feature_cols[j],
                'target_cat': categories[j],
                'weight'    : round(W[i, j], 4)
            })

edges_df = (
    pd.DataFrame(edges_list)
    .sort_values('weight', key=abs, ascending=False)
)

print(f'Total causal edges: {len(edges_df)}')
print()
print('Top 15 edges by weight:')
print(edges_df.head(15).to_string(index=False))

Total causal edges: 37

Top 15 edges by weight:
                        source source_cat            target target_cat  weight
                         crash   incident is_major_incident    traffic  0.8845
                 special_event   incident      impact_hours    traffic  0.5712
                        hazard   incident is_major_incident    traffic  0.3597
                     breakdown   incident      impact_hours    traffic  0.2632
            scheduled_roadwork   incident      impact_hours    traffic  0.2598
                         crash   incident      impact_hours    traffic  0.2084
            emergency_roadwork   incident      impact_hours    traffic  0.1606
                   degree_cent       meta  mean_flow_hourly    traffic  0.1227
    traffic_lights_blacked_out   incident      impact_hours    traffic  0.1189
                        hazard   incident      impact_hours    traffic  0.1035
               adverse_weather   incident      impact_hours    traffic  0.0908
    

## 8. Visualise causal graph (TraffiDent Fig. 5(a) style, sparsified)

Layered DAG on a white background, following the TraffiDent paper:
- Light-blue nodes (top) = meta-features · light-green (middle) = incident variables · light-yellow (bottom) = traffic statistics
- Thick solid = strong, thin solid = regular, thin dashed = weak causal relations

To keep the figure readable (the paper's MM-DAG output is very sparse and its
figure is hand-polished), we keep only the **top 3 strongest parents per node**
and cap the total at **30 edges**; isolated nodes are hidden. Adjust
`TOP_K_PARENTS` / `MAX_EDGES` to show more or fewer edges. The drawn edge list
is exported to `dag_edges_for_drawio.csv` for optional manual refinement.


In [ ]:
# ── TraffiDent Figure 5(a)-style causal DAG (clean version) ──────────────
# Key differences from a raw plot of ALL Lasso edges:
#   1. Keep only the TOP_K_PARENTS strongest parents per target node
#      (mimics MM-DAG's L1 sparsity — the paper keeps very few edges)
#   2. Cap the total number of drawn edges at MAX_EDGES
#   3. Hide nodes that end up with no edges
#   4. Shorten long labels so boxes don't overlap
# The paper's figure is additionally hand-tuned in a diagram editor;
# for a publication-final version, export edges and refine in draw.io.

EDGE_THRESH   = 0.01
TOP_K_PARENTS = 3     # strongest parents kept per target
MAX_EDGES     = 30    # global cap on drawn edges
LABEL_MAXLEN  = 16    # truncate long variable names

sig = edges_df[abs(edges_df['weight']) > EDGE_THRESH].copy()
sig['abs_w'] = sig['weight'].abs()

# 1) per-target top-k parents  2) global top MAX_EDGES
sig = (sig.sort_values('abs_w', ascending=False)
          .groupby('target', group_keys=False)
          .head(TOP_K_PARENTS)
          .sort_values('abs_w', ascending=False)
          .head(MAX_EDGES))

print(f'Edges drawn: {len(sig)} '
      f'(of {len(edges_df)} above threshold — rest hidden for clarity)')

# Only nodes that participate in a drawn edge
used = set(sig['source']) | set(sig['target'])

G_dag = nx.DiGraph()
for i, var in enumerate(feature_cols):
    if var in used:
        G_dag.add_node(var, category=categories[i])
for _, row in sig.iterrows():
    G_dag.add_edge(row['source'], row['target'], weight=row['weight'])

# ── Node colours (TraffiDent / draw.io palette) ──────────────────────────
cat_face = {'meta': '#dae8fc', 'incident': '#d5e8d4', 'traffic': '#fff2cc'}
cat_edge = {'meta': '#6c8ebf', 'incident': '#82b366', 'traffic': '#d6b656'}

def short(name, n=LABEL_MAXLEN):
    return name if len(name) <= n else name[:n - 1] + '…'

# ── Layered positions: meta top / incident middle / traffic bottom ───────
def layer_positions(names, y0, y1=None, max_per_row=6):
    pos = {}
    n = len(names)
    if n == 0:
        return pos
    xs = np.linspace(0.0, 1.0, n) if n > 1 else [0.5]
    for k, name in enumerate(names):
        y = y0 if (y1 is None or n <= max_per_row or k % 2 == 0) else y1
        pos[name] = (xs[k], y)
    return pos

meta_nodes     = [v for v in feature_cols
                  if v in used and categories[feature_cols.index(v)] == 'meta']
incident_nodes = [v for v in feature_cols
                  if v in used and categories[feature_cols.index(v)] == 'incident']
traffic_nodes  = [v for v in feature_cols
                  if v in used and categories[feature_cols.index(v)] == 'traffic']

pos = {}
pos.update(layer_positions(meta_nodes,     y0=2.35, y1=2.05))
pos.update(layer_positions(incident_nodes, y0=1.35, y1=1.05))
pos.update(layer_positions(traffic_nodes,  y0=0.20))

# ── Edge strength tiers ──────────────────────────────────────────────────
abs_w = sig['abs_w']
q_hi, q_lo = abs_w.quantile(0.67), abs_w.quantile(0.33)

def edge_style(w):
    if abs(w) >= q_hi:
        return dict(lw=2.4, ls='-',  color='#111111', alpha=0.95)  # strong
    elif abs(w) >= q_lo:
        return dict(lw=1.1, ls='-',  color='#444444', alpha=0.85)  # regular
    else:
        return dict(lw=0.9, ls='--', color='#999999', alpha=0.75)  # weak

fig, ax = plt.subplots(figsize=(13, 9))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Edges first; alternate arc direction/curvature to reduce overlap
for k, (u, v, d) in enumerate(G_dag.edges(data=True)):
    st  = edge_style(d['weight'])
    rad = 0.12 + 0.06 * (k % 3)
    if k % 2 == 1:
        rad = -rad
    ax.annotate(
        '', xy=pos[v], xytext=pos[u],
        arrowprops=dict(
            arrowstyle='-|>', shrinkA=16, shrinkB=16,
            connectionstyle=f'arc3,rad={rad}',
            lw=st['lw'], linestyle=st['ls'],
            color=st['color'], alpha=st['alpha'],
        ),
        zorder=1,
    )

# Nodes as rounded boxes
for name in G_dag.nodes:
    cat = G_dag.nodes[name]['category']
    ax.text(
        pos[name][0], pos[name][1], short(name),
        ha='center', va='center', fontsize=8.5, color='black',
        bbox=dict(boxstyle='round,pad=0.35',
                  fc=cat_face[cat], ec=cat_edge[cat], lw=1.2),
        zorder=3,
    )

legend_items = [
    mpatches.Patch(fc=cat_face['meta'],     ec=cat_edge['meta'],
                   label='Meta-features'),
    mpatches.Patch(fc=cat_face['incident'], ec=cat_edge['incident'],
                   label='Incident Variable'),
    mpatches.Patch(fc=cat_face['traffic'],  ec=cat_edge['traffic'],
                   label='Traffic Statistic'),
    plt.Line2D([0], [0], color='#111111', lw=2.4,
               label='Strong causal relation'),
    plt.Line2D([0], [0], color='#444444', lw=1.1,
               label='Regular causal relation'),
    plt.Line2D([0], [0], color='#999999', lw=0.9, ls='--',
               label='Weak causal relation'),
]
ax.legend(handles=legend_items, loc='lower center', ncol=3,
          bbox_to_anchor=(0.5, -0.10), frameon=False, fontsize=9)

ax.set_title(
    'NSW Global Causal Network (Lasso-based, following MM-DAG)\n'
    f'Top {TOP_K_PARENTS} parents per node · {len(sig)} strongest edges shown',
    fontsize=12, color='black', pad=12,
)
ax.set_xlim(-0.10, 1.10)
ax.set_ylim(-0.15, 2.65)
ax.axis('off')

plt.tight_layout()
plt.savefig('results/global_causal/global_causal_graph.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: global_causal_graph.png')

# Export the drawn edge list — import into draw.io for a hand-polished
# publication figure, exactly like TraffiDent did.
sig.drop(columns='abs_w').to_csv(
    'results/global_causal/dag_edges_for_drawio.csv', index=False)
print('Saved: dag_edges_for_drawio.csv (for manual layout refinement)')


## 9. Visualise causal weight heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

vmax = np.abs(W).max()
im   = ax.imshow(W, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

ax.set_xticks(range(n_vars))
ax.set_yticks(range(n_vars))
ax.set_xticklabels(feature_cols, rotation=45, ha='right',
                   color='black', fontsize=8)
ax.set_yticklabels(feature_cols, color='black', fontsize=8)

cb = plt.colorbar(im, ax=ax, fraction=0.03)
cb.set_label('Causal weight', color='black')
cb.ax.yaxis.set_tick_params(color='black')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='black')

ax.set_title('Global Causal Weight Matrix\nRow → causes → Column',
             color='black', fontsize=12)
for spine in ax.spines.values():
    spine.set_edgecolor('#888888')

plt.tight_layout()
plt.savefig('results/global_causal/global_causal_heatmap.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: global_causal_heatmap.png')


## 9b. Factual explanation of causal edges (TraffiDent Fig. 5(b) style)

For selected causal relationships $X \rightarrow Y$, plot the hourly average of
the outcome under different factor levels with a 95% confidence-interval band:
- (b1) Time (weekday/weekend) → incidents
- (b2) Weather (rain level) → incidents
- (b3) Incident type → hourly pattern
- (b4) Node role (hub/fringe by degree centrality) → incidents

**Requires:** `final_incidents_with_weather.csv`; panel (b4) also needs
`centrality_rankings.csv` from the graph notebook.


In [ ]:
# ── TraffiDent Figure 5(b)-style factual explanation of causal edges ─────
# For each learned edge X → Y, plot the hourly average of outcome Y under
# different levels of factor X, with a 95% confidence-interval band
# (mean ± 1.96 · SEM across days), exactly like Fig. 5(b1)–(b4).

haz = pd.read_csv('final_incidents_with_weather.csv', low_memory=False)
haz['dt']   = pd.to_datetime(haz['match_hour'], errors='coerce')
haz         = haz.dropna(subset=['dt'])
haz['hour'] = haz['dt'].dt.hour
haz['date'] = haz['dt'].dt.date
haz['is_weekend'] = haz['dt'].dt.dayofweek >= 5

print('Columns available:', list(haz.columns))

# Style constants (matching the paper panels)
PANEL_STYLES = [
    dict(color='#1f77b4', marker='D'),   # blue diamond
    dict(color='#ff7f0e', marker='o'),   # orange circle
    dict(color='#2ca02c', marker='^'),   # green triangle
    dict(color='#d62728', marker='v'),   # red triangle-down
]

def hourly_mean_ci(df):
    """Per-hour mean incident count across days, with 95% CI."""
    daily = (df.groupby(['date', 'hour']).size()
               .rename('cnt').reset_index())
    # Include zero-incident hours so the mean is not inflated
    all_days  = daily['date'].unique()
    full_idx  = pd.MultiIndex.from_product([all_days, range(24)],
                                           names=['date', 'hour'])
    daily = (daily.set_index(['date', 'hour'])
                  .reindex(full_idx, fill_value=0).reset_index())
    g    = daily.groupby('hour')['cnt']
    mean = g.mean()
    sem  = g.std() / np.sqrt(g.count())
    return mean, 1.96 * sem

def plot_factor_panel(ax, groups, title, ylabel):
    """groups = list of (label, sub_dataframe)."""
    for k, (label, sub) in enumerate(groups):
        if len(sub) == 0:
            continue
        st = PANEL_STYLES[k % len(PANEL_STYLES)]
        mean, ci = hourly_mean_ci(sub)
        ax.plot(mean.index, mean.values, marker=st['marker'],
                ms=4, lw=1.3, color=st['color'], label=label)
        ax.fill_between(mean.index, mean - ci, mean + ci,
                        color=st['color'], alpha=0.15)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(range(0, 24, 5))
    ax.grid(True, linestyle='--', linewidth=0.5, color='#d9d9d9')
    ax.set_axisbelow(True)
    ax.legend(fontsize=8, framealpha=0.9, edgecolor='#888888')
    for spine in ax.spines.values():
        spine.set_edgecolor('#888888')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor('white')

# ── (b1) Time → Incident: weekdays vs weekends ───────────────────────────
plot_factor_panel(
    axes[0, 0],
    [('Weekday', haz[~haz['is_weekend']]),
     ('Weekend', haz[haz['is_weekend']])],
    '(b1) Average incident trends: Weekdays vs Weekends',
    'Average Incident Count',
)

# ── (b2) Weather → Incident: rain levels ─────────────────────────────────
# Auto-detect a precipitation / rain / weather column
weather_col = next(
    (c for c in haz.columns
     if any(k in c.lower() for k in ('precip', 'rain', 'weather'))
     and c.lower() != 'match_hour'),
    None,
)
if weather_col is not None and pd.api.types.is_numeric_dtype(haz[weather_col]):
    w = haz[weather_col].fillna(0)
    groups_b2 = [
        ('No Rain',    haz[w <= 0]),
        ('Light Rain', haz[(w > 0) & (w <= w[w > 0].median())]),
        ('Heavy Rain', haz[w > w[w > 0].median()]),
    ]
elif weather_col is not None:
    top_levels = haz[weather_col].value_counts().head(3).index
    groups_b2  = [(str(l), haz[haz[weather_col] == l]) for l in top_levels]
else:
    groups_b2 = []
    print('[note] No weather column detected — panel (b2) left empty. '
          'Set weather_col manually if needed.')

plot_factor_panel(
    axes[0, 1], groups_b2,
    f'(b2) Incident trends under different weather ({weather_col})',
    'Average Incident Count',
)

# ── (b3) Incident type → hourly pattern: top-3 hazard types ──────────────
top_types = haz['hazard_type'].value_counts().head(3).index
plot_factor_panel(
    axes[1, 0],
    [(str(t), haz[haz['hazard_type'] == t]) for t in top_types],
    '(b3) Hourly trends of top-3 incident types',
    'Average Incident Count',
)

# ── (b4) Node role → Incident: hub vs fringe stations ────────────────────
# Uses the centrality rankings exported by the graph notebook.
try:
    cent = pd.read_csv('centrality_rankings.csv')
    cent['station_id'] = cent['station_id'].astype(str).str.strip()
    haz['station_id']  = haz['station_id'].astype(str).str.strip()
    K = 20   # NSW network has 115 stations → top/bottom 20 by degree
    hub_ids    = set(cent.nlargest(K,  'degree_cent')['station_id'])
    fringe_ids = set(cent.nsmallest(K, 'degree_cent')['station_id'])
    plot_factor_panel(
        axes[1, 1],
        [('Hub Node',    haz[haz['station_id'].isin(hub_ids)]),
         ('Fringe Node', haz[haz['station_id'].isin(fringe_ids)])],
        f'(b4) Incident trends on hub vs fringe nodes (top/bottom {K} by degree)',
        'Average Incident Count',
    )
except FileNotFoundError:
    axes[1, 1].axis('off')
    print('[note] centrality_rankings.csv not found — panel (b4) skipped. '
          'Run the graph/centrality notebook first.')

fig.suptitle(
    'Factual explanation of learned causal edges '
    '(mean ± 95% CI across days)',
    fontsize=12, y=1.00,
)
plt.tight_layout()
plt.savefig('results/global_causal/global_causal_factual_explanation.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: global_causal_factual_explanation.png')


## 10. Export significant causal links

In [23]:
sig_links = edges_df[abs(edges_df['weight']) > EDGE_THRESH].copy()
sig_links['direction']  = sig_links['weight'].apply(
    lambda w: 'positive' if w > 0 else 'negative'
)
sig_links['abs_weight'] = sig_links['weight'].abs()
sig_links = sig_links.sort_values('abs_weight', ascending=False)

print(f'Significant causal links: {len(sig_links)}')
print(sig_links.to_string(index=False))

sig_links.to_csv(
    'results/global_causal/significant_causal_links.csv', index=False
)
print('\nSaved: significant_causal_links.csv')

Significant causal links: 37
                        source source_cat                     target target_cat  weight direction  abs_weight
                         crash   incident          is_major_incident    traffic  0.8845  positive      0.8845
                 special_event   incident               impact_hours    traffic  0.5712  positive      0.5712
                        hazard   incident          is_major_incident    traffic  0.3597  positive      0.3597
                     breakdown   incident               impact_hours    traffic  0.2632  positive      0.2632
            scheduled_roadwork   incident               impact_hours    traffic  0.2598  positive      0.2598
                         crash   incident               impact_hours    traffic  0.2084  positive      0.2084
            emergency_roadwork   incident               impact_hours    traffic  0.1606  positive      0.1606
                   degree_cent       meta           mean_flow_hourly    traffic  0.1227  po

## 11. Causal links by category pair

In [24]:
cat_summary = (
    sig_links
    .groupby(['source_cat', 'target_cat'])
    .agg(
        n_links         = ('weight', 'count'),
        mean_abs_weight = ('abs_weight', 'mean')
    )
    .reset_index()
    .sort_values('n_links', ascending=False)
)

print('Causal links by category pair:')
print(cat_summary.to_string(index=False))

Causal links by category pair:
source_cat target_cat  n_links  mean_abs_weight
  incident    traffic       20         0.174720
  incident   incident        8         0.027275
      meta   incident        6         0.042217
      meta    traffic        3         0.087233


## 12. Summary

In [25]:
print('GLOBAL CAUSAL ANALYSIS — RESULTS SUMMARY')
print('=' * 50)
print(f'Algorithm           : DYNOTEARS')
print(f'Total variables     : {n_vars}')
print(f'Total samples       : {X.shape[0]}')
print(f'Edge threshold      : |weight| > {EDGE_THRESH}')
print(f'Significant links   : {len(sig_links)}')
print(f'Positive effects    : {(sig_links["direction"]=="positive").sum()}')
print(f'Negative effects    : {(sig_links["direction"]=="negative").sum()}')
print()
print('Top 5 strongest causal links:')
print(sig_links[['source','target','weight','direction']].head(5).to_string(index=False))
print()
print('Output files saved to: results/global_causal/')
print('  global_causal_graph.png')
print('  global_causal_heatmap.png')
print('  significant_causal_links.csv')

GLOBAL CAUSAL ANALYSIS — RESULTS SUMMARY
Algorithm           : DYNOTEARS
Total variables     : 26
Total samples       : 24455
Edge threshold      : |weight| > 0.01
Significant links   : 37
Positive effects    : 35
Negative effects    : 2

Top 5 strongest causal links:
            source            target  weight direction
             crash is_major_incident  0.8845  positive
     special_event      impact_hours  0.5712  positive
            hazard is_major_incident  0.3597  positive
         breakdown      impact_hours  0.2632  positive
scheduled_roadwork      impact_hours  0.2598  positive

Output files saved to: results/global_causal/
  global_causal_graph.png
  global_causal_heatmap.png
  significant_causal_links.csv
